# The AI Telco Troubleshooting Challenge

Goal: Enhance the accuracy of Qwen3-32B when answering telco troubleshooting questions in telelogs data.

## I/ Set-up

In [ ]:
!git clone https://github.com/Kodro23/Telco-challenge

Cloning into 'Telco-challenge'...
remote: Enumerating objects: 21, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 21 (delta 6), reused 4 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (21/21), 1.91 MiB | 3.96 MiB/s, done.
Resolving deltas: 100% (6/6), done.


### 1. Load libraries

In [ ]:
import pandas as pd
import numpy as np
import re

### 2.Load data

In [ ]:
#load data
df = pd.read_csv("/content/Telco-challenge/train.csv")
print(df.head())

              ID                                           question answer
0  ID_1P7PJMPV0R  Analyze the 5G wireless network drive-test use...     C2
1  ID_8B1D1TUTFA  Analyze the 5G wireless network drive-test use...     C1
2  ID_IGGXMA9GZH  Analyze the 5G wireless network drive-test use...     C2
3  ID_D6C9N2X295  Analyze the 5G wireless network drive-test use...     C2
4  ID_8JC15PNP3Q  Analyze the 5G wireless network drive-test use...     C5


In [ ]:
#Function to clean questions
def clean_question(question):
    lines=question.split("\n")
    content=[]
    for line in lines:
        if "|" in line:
            content.append(line.split("|"))
    drive_test_data=[{"Observation": i+1,**dict(zip(content[0],row))} for i,row in enumerate(content[1:])]
    engineering_params=[{"Observation": i+1,**dict(zip(content[11],row))} for i,row in enumerate(content[12:])]
    #clean question
    ##recompute begining of the prompt
    q=""
    for l in [l+"\n" for l in lines[:19]]:
        q=q+l
    ##assemble
    cleaned_question= f" Question: {q} \nDrive test data: {drive_test_data} \nEngineering parameters {engineering_params} "
    return cleaned_question
#Apply to dataset
df["cleaned_question"]=df["question"].apply(lambda x: clean_question(x))
print(df["cleaned_question"][0])

 Question: Analyze the 5G wireless network drive-test user plane data and engineering parameters.
Identify the reason for the throughput dropping below 600Mbps in certain road sections.
From the following 8 potential root causes, select the most likely one and enclose its number in \boxed{{}} in the final answer.

C1: The serving cell's downtilt angle is too large, causing weak coverage at the far end.
C2: The serving cell's coverage distance exceeds 1km, resulting in over-shooting.
C3: A neighboring cell provides higher throughput.
C4: Non-colocated co-frequency neighboring cells cause severe overlapping coverage.
C5: Frequent handovers degrade performance.
C6: Neighbor cell and serving cell have the same PCI mod 30, leading to interference.
C7: Test vehicle speed exceeds 40km/h, impacting user throughput.
C8: Average scheduled RBs are below 160, affecting throughput.

Given:
- The default electronic downtilt value is 255, representing a downtilt angle of 6 degrees. Other values repre

## II/ Untrained model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:

# load the tokenizer and the model
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
def find_the_root_cause(question):
  messages = [
    {"role": "user", "content": question}
  ]
  text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
    )
  model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
  # conduct text completion
  generated_ids = model.generate(
      **model_inputs,
      max_new_tokens=1024
      )
  output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
  # parsing thinking content
  try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
  except ValueError:
    index = 0
    content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

  match = re.search(r"\\boxed\{(C\d)\}", content)
  if match:
      return match.group(1)
  else:
      return "UNKNOWN"


In [ ]:
df["root_cause"]=df["cleaned_question"].apply(lambda x: find_the_root_cause(x))

OutOfMemoryError: CUDA out of memory. Tried to allocate 1.07 GiB. GPU 0 has a total capacity of 14.56 GiB of which 943.81 MiB is free. Including non-PyTorch memory, this process has 13.64 GiB memory in use. Of the allocated memory 12.38 GiB is allocated by PyTorch, and 1.14 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)